# Exercise 3: SelectPercentile on iris

In [1]:
import sys
from pathlib import Path

import numpy as np

# find the repository root (the folder that contains "datasets"), wherever the notebook is
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
for path in (ROOT / 'src', ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA = ROOT / 'datasets'

import pandas as pd
from si.io.csv_file import read_csv
from si.feature_selection.select_percentile import SelectPercentile
from si.statistics.f_classification import f_classification

In [2]:
iris = read_csv(str(DATA / 'iris' / 'iris.csv'), features=True, label=True)
selector = SelectPercentile(score_func=f_classification, percentile=50)
selector.fit(iris)
pd.DataFrame({'F': selector.F, 'p-value': selector.p}, index=iris.features).sort_values('F', ascending=False)

,F,p-value
petal_length,1179.034328,3.051976e-91
petal_width,959.324406,4.376957e-85
sepal_length,119.264502,1.669669e-31
sepal_width,47.364461,1.327917e-16


50% of 4 features = 2 features: the two with the highest F (the petal measurements separate the species best).

In [3]:
new_iris = selector.transform(iris)
print(new_iris.shape(), new_iris.features)

(150, 2) ['petal_length', 'petal_width']


## Different percentiles

In [4]:
for pct in [25, 50, 75, 100]:
    print(pct, SelectPercentile(percentile=pct).fit_transform(iris).features)

25 ['petal_length']
50 ['petal_length', 'petal_width']
75 ['sepal_length', 'petal_length', 'petal_width']
100 ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


## Ties: the example from the slides
F = [1.2, 3.4, 2.1, 5.6, 4.3, 5.6, 7.8, 6.5, 5.6, 3.2], percentile = 40 -> 4 features. Only 7.8 and 6.5 are above the threshold (5.6), so the first two features tied at 5.6 are added.

In [5]:
from si.data.dataset import Dataset
F = np.array([1.2, 3.4, 2.1, 5.6, 4.3, 5.6, 7.8, 6.5, 5.6, 3.2])
ds = Dataset(np.random.rand(6, 10), np.array([0, 1, 0, 1, 0, 1]))
sel = SelectPercentile(score_func=lambda d: (F, np.zeros_like(F)), percentile=40).fit(ds)
out = sel.transform(ds)
print('threshold:', np.percentile(F, 60))
print(out.features, F[[int(f.split('_')[1]) for f in out.features]])

threshold: 5.6
['feat_3', 'feat_5', 'feat_6', 'feat_7'] [5.6 5.6 7.8 6.5]
